# nanoGPT + Prism Smoke Test
**Runtime → A100 or T4**

Tests Prism spectral init on nanoGPT using Shakespeare (tiny, fast).
Baseline vs Prism, 1000 steps each, ~5 min per run on A100.

If this works, move to OpenWebText for the real benchmark.

In [ ]:
# Cell 1: Clone + Setup
import os, subprocess
if os.path.exists('/content/nanogpt-prism'):
    subprocess.run(['rm', '-rf', '/content/nanogpt-prism'])
!git clone https://github.com/timepointai/nanogpt-prism-shakespeare.git /content/nanogpt-prism
%cd /content/nanogpt-prism
!pip install -q transformers tiktoken datasets
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')

# Prepare Shakespeare dataset (tiny, instant)
!python data/shakespeare_char/prepare.py
print('Shakespeare dataset ready.')

# Pre-cache GPT-2 for spectral extraction
from transformers import GPT2LMHeadModel
GPT2LMHeadModel.from_pretrained('gpt2')
print('GPT-2 cached. Ready.')

In [ ]:
# Cell 2: Run BASELINE (standard nanoGPT init, 1000 steps)
import subprocess, json, time

print('='*60)
print('  BASELINE: Standard Normal(0, 0.02) init')
print('='*60)

t0 = time.time()
result = subprocess.run([
    'python', 'train.py',
    'config/train_shakespeare_char.py',
    '--max_iters=1000',
    '--eval_interval=250',
    '--eval_iters=50',
    '--log_interval=50',
    '--out_dir=out-baseline',
    '--wandb_log=False',
    '--compile=False',
    '--prism_init=False',
], capture_output=True, text=True, timeout=600)

print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

elapsed = time.time() - t0
print(f'\nBaseline done in {elapsed:.0f}s')

# Save stdout for parsing
with open('/content/baseline_log.txt', 'w') as f:
    f.write(result.stdout)
print('SAVED: /content/baseline_log.txt')

In [ ]:
# Cell 3: Run PRISM (spectral init + EigenTransfer, 1000 steps)
import subprocess, time

print('='*60)
print('  PRISM: Spectral Imprint + EigenTransfer (align=0.75)')
print('='*60)

t0 = time.time()
result = subprocess.run([
    'python', 'train.py',
    'config/train_shakespeare_char.py',
    '--max_iters=1000',
    '--eval_interval=250',
    '--eval_iters=50',
    '--log_interval=50',
    '--out_dir=out-prism',
    '--wandb_log=False',
    '--compile=False',
    '--prism_init=True',
    '--prism_align=0.75',
], capture_output=True, text=True, timeout=600)

print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

elapsed = time.time() - t0
print(f'\nPrism done in {elapsed:.0f}s')

with open('/content/prism_log.txt', 'w') as f:
    f.write(result.stdout)
print('SAVED: /content/prism_log.txt')

In [ ]:
# Cell 4: Parse and compare val losses
import re

def parse_val_losses(log_path):
    """Extract (step, val_loss) pairs from nanoGPT output."""
    losses = {}
    with open(log_path) as f:
        for line in f:
            # nanoGPT logs: "step N: train loss X.XXXX, val loss Y.YYYY"
            m = re.search(r'step (\d+):.*val loss ([\d.]+)', line)
            if m:
                losses[int(m.group(1))] = float(m.group(2))
    return losses

baseline = parse_val_losses('/content/baseline_log.txt')
prism = parse_val_losses('/content/prism_log.txt')

print('='*50)
print('  nanoGPT + Prism: Shakespeare Smoke Test')
print('='*50)
print(f'\n{"Step":>6s}  {"Baseline":>10s}  {"Prism":>10s}  {"Ratio":>8s}')
print('-' * 40)

all_steps = sorted(set(baseline.keys()) | set(prism.keys()))
for step in all_steps:
    b = baseline.get(step)
    p = prism.get(step)
    if b and p:
        # Lower val loss = better. Ratio > 1 means Prism is better.
        ratio = f'{b/p:.2f}x' if p > 0 else 'n/a'
        print(f'{step:>6d}  {b:>10.4f}  {p:>10.4f}  {ratio:>8s}')
    elif b:
        print(f'{step:>6d}  {b:>10.4f}  {"—":>10s}')
    elif p:
        print(f'{step:>6d}  {"—":>10s}  {p:>10.4f}')

if baseline and prism:
    last_b = baseline[max(baseline.keys())]
    last_p = prism[max(prism.keys())]
    print(f'\nFinal: baseline={last_b:.4f}  prism={last_p:.4f}')
    if last_p < last_b:
        print(f'Prism wins by {last_b - last_p:.4f} val loss ({last_b/last_p:.2f}x)')
    else:
        print(f'Baseline wins by {last_p - last_b:.4f} val loss')
else:
    print('\nCould not parse results — check logs above')

In [ ]:
# Cell 5: Download logs
from google.colab import files
for f in ['/content/baseline_log.txt', '/content/prism_log.txt']:
    try: files.download(f)
    except: print(f'{f} not found')